In [ ]:
# %pip install transformers evaluate torch peft
# %pip install accelerate
# %pip install torchao==0.16.0

In [ ]:
import torch
from transformers import RobertaTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model
import evaluate
from datasets import Dataset
from peft import AutoPeftModelForSequenceClassification
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
import json


In [ ]:
frozen_train = pd.read_csv('train_dataset.csv')
frozen_val = pd.read_csv('val_dataset.csv')
frozen_test = pd.read_csv('test_dataset.csv')

# Load model and basic preprocessing

In [ ]:
base_model = 'roberta-base'
OUTPUT_DIR = 'roberta_configs'
peft_model_name = f'{OUTPUT_DIR}/peft_model'
tokenizer_dir = f'{OUTPUT_DIR}/tokenizer'
tokenized_test_dir = f'{OUTPUT_DIR}/tokenized_test'
tokenized_val_dir = f'{OUTPUT_DIR}/tokenized_val'

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained(base_model)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [21]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

In [22]:
def to_tokenized(df):
    ds = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
    return ds.map(preprocess_function, batched=True, remove_columns=['text'])

In [23]:
tokenized_train_dataset = to_tokenized(frozen_train)
tokenized_val_dataset = to_tokenized(frozen_val)
tokenized_test_dataset = to_tokenized(frozen_test)

Map:   0%|          | 0/65401 [00:00<?, ? examples/s]

Map:   0%|          | 0/8218 [00:00<?, ? examples/s]

Map:   0%|          | 0/8151 [00:00<?, ? examples/s]

In [ ]:
# save tokenized datasets
tokenized_test_dataset.rename_column("label", "labels").save_to_disk(tokenized_test_dir)
tokenized_val_dataset.rename_column("label", "labels").save_to_disk(tokenized_val_dir)
frozen_test.to_csv(f'{OUTPUT_DIR}/test_dataset.csv', index=False)

In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

# Training

In [12]:
id2label = {0: 'human', 1: 'LLM-generated'}


In [29]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='steps',
    eval_steps=200,
    learning_rate=2e-4,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    logging_steps=50,
)

In [30]:
def get_trainer(model):
      return  Trainer(
          model=model,
          args=training_args,
          train_dataset=tokenized_train_dataset,
          eval_dataset=tokenized_val_dataset,
          data_collator=data_collator,
      )

In [31]:
peft_config = LoraConfig(task_type="SEQ_CLS",
                         inference_mode=False,
                         r=8,
                         lora_alpha=16,
                         lora_dropout=.1)


In [33]:
model = AutoModelForSequenceClassification.from_pretrained(base_model, id2label=id2label)

peft_model = get_peft_model(model, peft_config)

print('PEFT Model')
peft_model.print_trainable_parameters()

peft_lora_finetuning_trainer = get_trainer(peft_model)

peft_lora_finetuning_trainer.train()
val_metrics = peft_lora_finetuning_trainer.evaluate()
val_metrics

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PEFT Model
trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


Step,Training Loss,Validation Loss
200,0.112487,0.187074
400,0.057080,0.110626
600,0.045560,0.068231
800,0.063132,0.153513
1000,0.016948,0.045991
1200,0.014421,0.035540
1400,0.129275,0.111252
1600,0.020533,0.119145
1800,0.071938,0.344821
2000,0.075876,0.243757


Training Loss,Validation Loss,Step
0.000050,0.074475,4088


{'eval_loss': 0.0744745060801506}

# Save the tokenizer & the model weights

In [ ]:
tokenizer.save_pretrained(tokenizer_dir)
peft_model.save_pretrained(peft_model_name)

with open(f'{OUTPUT_DIR}/val_metrics.json', 'w') as f:
    json.dump(val_metrics, f, indent=2)
val_metrics

# Evaluate

In [ ]:
DATA_PATH = "./../data/"
ROBERTA_CONFIGS_PATHS = DATA_PATH + "roberta_configs/"
FROZEN_TEST_PATH = ROBERTA_CONFIGS_PATHS + "test_dataset.csv"
ROBERTA_WEIGHTS_PATH = ROBERTA_CONFIGS_PATHS + "peft_model"
TOKENIZER_PATH = ROBERTA_CONFIGS_PATHS + "tokenizer"
TOKENIZED_TEST_PATH = ROBERTA_CONFIGS_PATHS + "tokenized_test"
id2label = {0: 'human', 1: 'LLM-generated'}


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

In [ ]:
import numpy as np

In [ ]:
inference_model = AutoPeftModelForSequenceClassification.from_pretrained(ROBERTA_WEIGHTS_PATH, id2label=id2label)

In [ ]:
def _metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )
    return {
        "macro_f1": float(f1.mean()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "per_class": pd.DataFrame(
            {"precision": precision, "recall": recall, "f1": f1, "support": support},
            index=pd.Index(["human", "ai"], name="class"),
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]),
    }

In [ ]:
def evaluate_model(inference_model, dataset, sources, batch_size=64):
    eval_dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    inference_model.to(device)
    inference_model.eval()
    y_true, y_pred = [], []
    for batch in tqdm(eval_dataloader):
        batch = batch.to(device)
        with torch.no_grad():
            outputs = inference_model(**batch)
        y_pred.append(outputs.logits.argmax(dim=-1))
        y_true.append(batch["labels"])

    truth = torch.cat(y_true).cpu().numpy()
    predictions = torch.cat(y_pred).cpu().numpy()
    domains = np.asarray(sources)
    result = _metrics(truth, predictions)
    result["per_domain"] = {
        source: _metrics(truth[domains == source], predictions[domains == source])
        for source in sorted(set(domains))
    }
    return result

In [ ]:
scores = evaluate_model(inference_model, tokenized_test_dataset, frozen_test["source"])
print(scores["macro_f1"], scores["accuracy"])
scores["per_class"]

pasting in results from g collab here for book keeping

100%|██████████| 128/128 [00:30<00:00,  4.21it/s]0.9902111550799262 0.9914120966752545
	precision	recall	f1	support
class				
human	0.999634	0.987717	0.993640	5536
ai	0.974636	0.999235	0.986782	2615